# Silver Cleaning (A): sifariş axını

**Məsuliyyətli:** Nəfər A

| Bronze | Silver |
|---|---|
| `bronze.olist_customers_dataset` | `silver.customers` |
| `bronze.olist_orders_dataset` | `silver.orders` |
| `bronze.olist_order_items_dataset` | `silver.order_items` |
| `bronze.olist_order_payments_dataset` | `silver.order_payments` |
| `bronze.api_holidays` | `silver.holidays` |




In [1]:
import sys
sys.path.append("/workspaces/olist-ecommerce-analytics/notebooks")

import pandas as pd
from db import get_engine

In [2]:
import time
from datetime import date, timedelta
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError
from sqlalchemy.engine import URL
import sys
sys.path.append("/workspaces/olist-ecommerce-analytics/notebooks")
from db import get_engine
import os
import json


In [3]:
df = pd.read_sql("SELECT * FROM bronze.olist_customers_dataset", get_engine())

print("Ölçü:", df.shape)
print("\nSütunlar və tiplər:")
print(df.dtypes)
print("\nBoş mətn (\"\") sayı:")
print((df == "").sum())
print("\nNULL sayı:")
print(df.isna().sum())
print("\nTam təkrarlanan sətir:", df.duplicated().sum())
print("customer_id təkrarı:", df["customer_id"].duplicated().sum())
print("\nİlk 5 sətir:")
df.head()

Ölçü: (99441, 5)

Sütunlar və tiplər:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix    object
customer_city               object
customer_state              object
dtype: object

Boş mətn ("") sayı:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

NULL sayı:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Tam təkrarlanan sətir: 0
customer_id təkrarı: 0

İlk 5 sətir:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [4]:
print("Fərqli customer_id:", df["customer_id"].nunique())
print("Fərqli customer_unique_id:", df["customer_unique_id"].nunique())

dup = df[df["customer_unique_id"].duplicated(keep=False)].sort_values("customer_unique_id")
dup.head(6)

Fərqli customer_id: 99441
Fərqli customer_unique_id: 96096


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
35608,24b0e2bd287e47d54d193e7bbb51103f,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
19299,1afe8a9c67eec3516c09a8bdcc539090,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
20023,1b4a75b3478138e99902678254b260f4,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ
22066,f6efe5d5c7b85e12355f9d5c3db46da2,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ
72451,49cf243e0d353cd418ca77868e24a670,004b45ec5c64187465168251cd1c9c2f,57055,maceio,AL
87012,d95f60d70d9ea9a7fe37c53c931940bb,004b45ec5c64187465168251cd1c9c2f,57035,maceio,AL


In [5]:
print("Ştat sayı:", df["customer_state"].nunique())
print("Ştat uzunluqları:", df["customer_state"].str.len().value_counts().to_dict())
print("Zip uzunluqları:", df["customer_zip_code_prefix"].str.len().value_counts().to_dict())
print("Fərqli şəhər sayı:", df["customer_city"].nunique())
print("Kənarında boşluq olan şəhər:", (df["customer_city"] != df["customer_city"].str.strip()).sum())
print("Kiçik hərflə olmayan şəhər:", (df["customer_city"] != df["customer_city"].str.lower()).sum())

Ştat sayı: 27
Ştat uzunluqları: {2: 99441}
Zip uzunluqları: {5: 99441}
Fərqli şəhər sayı: 4119
Kənarında boşluq olan şəhər: 0
Kiçik hərflə olmayan şəhər: 0


In [6]:
from sqlalchemy import types as sqltypes

customers = pd.read_sql("SELECT * FROM bronze.olist_customers_dataset", get_engine())

customers.to_sql(
    "customers",
    get_engine(),
    schema="silver",
    if_exists="replace",
    index=False,
    dtype={
        "customer_id": sqltypes.VARCHAR(32),
        "customer_unique_id": sqltypes.VARCHAR(32),
        "customer_zip_code_prefix": sqltypes.CHAR(5),
        "customer_city": sqltypes.VARCHAR(100),
        "customer_state": sqltypes.CHAR(2),
    },
)

138

In [7]:
print("silver.customers sətir sayı:", pd.read_sql("SELECT COUNT(*) AS n FROM silver.customers", get_engine()).iloc[0, 0])

pd.read_sql("SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA = 'silver' AND TABLE_NAME = 'customers'", get_engine())

silver.customers sətir sayı: 99441


,COLUMN_NAME,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH
0,customer_id,varchar,32
1,customer_unique_id,varchar,32
2,customer_zip_code_prefix,char,5
3,customer_city,varchar,100
4,customer_state,char,2


**Orders

In [8]:
print("silver.customers sətir sayı:", pd.read_sql("SELECT COUNT(*) AS n FROM silver.customers", get_engine()).iloc[0, 0])
print("-----")

orders = pd.read_sql("SELECT * FROM bronze.olist_orders_dataset", get_engine())

print("Ölçü:", orders.shape)
print("\nBoş mətn (\"\") sayı:")
print((orders == "").sum())
print("\nNULL sayı:")
print(orders.isna().sum())
print("\nTam təkrar sətir:", orders.duplicated().sum())
print("order_id təkrarı:", orders["order_id"].duplicated().sum())
print("\nSifariş statusları:")
print(orders["order_status"].value_counts())
orders.head()

silver.customers sətir sayı: 99441
-----


Ölçü: (99441, 8)

Boş mətn ("") sayı:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

NULL sayı:
order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

Tam təkrar sətir: 0
order_id təkrarı: 0

Sifariş statusları:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [9]:
print("Tam təkrar sətir:", orders.duplicated().sum())
print("order_id təkrarı:", orders["order_id"].duplicated().sum())
print(orders["order_status"].value_counts().to_string())

Tam təkrar sətir: 0
order_id təkrarı: 0
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2


In [10]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

print("Tarix formatı yoxlaması:")
for c in date_cols:
    empty = (orders[c] == "").sum()
    parsed = pd.to_datetime(orders[c].mask(orders[c] == ""), format="%Y-%m-%d %H:%M:%S", errors="coerce")
    bad = parsed.isna().sum() - empty
    print(f"  {c} | boş: {empty} | tanınmayan format: {bad}")

print("\nStatus üzrə boş tarixlər:")
check = orders.assign(
    approved_empty=(orders["order_approved_at"] == ""),
    carrier_empty=(orders["order_delivered_carrier_date"] == ""),
    customer_empty=(orders["order_delivered_customer_date"] == ""),
)
print(check.groupby("order_status")[["approved_empty", "carrier_empty", "customer_empty"]].sum())

Tarix formatı yoxlaması:
  order_purchase_timestamp | boş: 0 | tanınmayan format: 0
  order_approved_at | boş: 160 | tanınmayan format: 0
  order_delivered_carrier_date | boş: 1783 | tanınmayan format: 0
  order_delivered_customer_date | boş: 2965 | tanınmayan format: 0
  order_estimated_delivery_date | boş: 0 | tanınmayan format: 0

Status üzrə boş tarixlər:
              approved_empty  carrier_empty  customer_empty
order_status                                               
approved                   0              2               2
canceled                 141            550             619
created                    5              5               5
delivered                 14              2               8
invoiced                   0            314             314
processing                 0            301             301
shipped                    0              0            1107
unavailable                0            609             609


In [11]:
from sqlalchemy import types as sqltypes
from sqlalchemy.dialects.mssql import DATETIME2

orders = pd.read_sql("SELECT * FROM bronze.olist_orders_dataset", get_engine())

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# boş mətn -> NULL, sonra tarix tipinə
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c].mask(orders[c] == ""), format="%Y-%m-%d %H:%M:%S")

orders.to_sql(
    "orders",
    get_engine(),
    schema="silver",
    if_exists="replace",
    index=False,
    dtype={
        "order_id": sqltypes.VARCHAR(32),
        "customer_id": sqltypes.VARCHAR(32),
        "order_status": sqltypes.VARCHAR(20),
        **{c: DATETIME2(precision=0) for c in date_cols},
    },
)

143

In [12]:
print(pd.read_sql("SELECT COUNT(*) AS n FROM silver.orders", get_engine()))

pd.read_sql("""
SELECT
  SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS approved_null,
  SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) AS carrier_null,
  SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS customer_null
FROM silver.orders
""", get_engine())


       n
0  99441


,approved_null,carrier_null,customer_null
0,160,1783,2965


In [13]:
items = pd.read_sql("SELECT * FROM bronze.olist_order_items_dataset", get_engine())

print("Ölçü:", items.shape)
print("Boş mətn sayı:\n", (items == "").sum().to_string())
print("Tam təkrar sətir:", items.duplicated().sum())
print("(order_id, order_item_id) təkrarı:", items.duplicated(subset=["order_id", "order_item_id"]).sum())

price = pd.to_numeric(items["price"], errors="coerce")
freight = pd.to_numeric(items["freight_value"], errors="coerce")
print("price oxunmayan:", price.isna().sum(), "| freight oxunmayan:", freight.isna().sum())
print("price min/max:", price.min(), price.max())
print("freight min/max:", freight.min(), freight.max())

ship = pd.to_datetime(items["shipping_limit_date"].mask(items["shipping_limit_date"] == ""), format="%Y-%m-%d %H:%M:%S", errors="coerce")
print("shipping_limit_date tanınmayan:", ship.isna().sum())

orders_ids = pd.read_sql("SELECT order_id FROM silver.orders", get_engine())["order_id"]
print("silver.orders-də olmayan order_id:", (~items["order_id"].isin(orders_ids)).sum())

Ölçü: (112650, 7)
Boş mətn sayı:
 order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
Tam təkrar sətir: 0
(order_id, order_item_id) təkrarı: 0
price oxunmayan: 0 | freight oxunmayan: 0
price min/max: 0.85 6735.0
freight min/max: 0.0 409.68
shipping_limit_date tanınmayan: 0
silver.orders-də olmayan order_id: 0


In [14]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [15]:
from sqlalchemy import types as sqltypes
from sqlalchemy.dialects.mssql import DATETIME2

items = pd.read_sql("SELECT * FROM bronze.olist_order_items_dataset", get_engine())

items["order_item_id"] = pd.to_numeric(items["order_item_id"])
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], format="%Y-%m-%d %H:%M:%S")
items["price"] = pd.to_numeric(items["price"])
items["freight_value"] = pd.to_numeric(items["freight_value"])

items.to_sql(
    "order_items",
    get_engine(),
    schema="silver",
    if_exists="replace",
    index=False,
    dtype={
        "order_id": sqltypes.VARCHAR(32),
        "order_item_id": sqltypes.SMALLINT(),
        "product_id": sqltypes.VARCHAR(32),
        "seller_id": sqltypes.VARCHAR(32),
        "shipping_limit_date": DATETIME2(precision=0),
        "price": sqltypes.DECIMAL(10, 2),
        "freight_value": sqltypes.DECIMAL(10, 2),
    },
)

226

In [16]:
print("Sətir sayı:", pd.read_sql("SELECT COUNT(*) AS n FROM silver.order_items", get_engine()).iloc[0, 0])

sql_sum = pd.read_sql("SELECT SUM(price) AS p, SUM(freight_value) AS f FROM silver.order_items", get_engine())
print("SQL cəmi (price, freight):", sql_sum.iloc[0].tolist())
print("Pandas cəmi (price, freight):", round(items["price"].sum(), 2), round(items["freight_value"].sum(), 2))

pd.read_sql("""
SELECT COLUMN_NAME, DATA_TYPE, NUMERIC_PRECISION, NUMERIC_SCALE, CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'silver' AND TABLE_NAME = 'order_items'
ORDER BY ORDINAL_POSITION
""", get_engine())

Sətir sayı: 112650
SQL cəmi (price, freight): [13591643.7, 2251909.54]
Pandas cəmi (price, freight): 13591643.7 2251909.54


,COLUMN_NAME,DATA_TYPE,NUMERIC_PRECISION,NUMERIC_SCALE,CHARACTER_MAXIMUM_LENGTH
0,order_id,varchar,NaN,NaN,32.0
1,order_item_id,smallint,5.0,0.0,NaN
2,product_id,varchar,NaN,NaN,32.0
3,seller_id,varchar,NaN,NaN,32.0
4,shipping_limit_date,datetime2,NaN,NaN,NaN
5,price,decimal,10.0,2.0,NaN
6,freight_value,decimal,10.0,2.0,NaN


In [17]:
pay = pd.read_sql("SELECT * FROM bronze.olist_order_payments_dataset", get_engine())

print("Ölçü:", pay.shape)
print("Boş mətn sayı:\n", (pay == "").sum().to_string())
print("Tam təkrar sətir:", pay.duplicated().sum())
print("(order_id, payment_sequential) təkrarı:", pay.duplicated(subset=["order_id", "payment_sequential"]).sum())

print("\nÖdəniş növləri:")
print(pay["payment_type"].value_counts().to_string())

seq = pd.to_numeric(pay["payment_sequential"], errors="coerce")
inst = pd.to_numeric(pay["payment_installments"], errors="coerce")
val = pd.to_numeric(pay["payment_value"], errors="coerce")
print("\nOxunmayan -> sequential:", seq.isna().sum(), "| installments:", inst.isna().sum(), "| value:", val.isna().sum())
print("installments min/max:", inst.min(), inst.max())
print("value min/max:", val.min(), val.max())
print("value = 0 olan sətir:", (val == 0).sum())
print("installments = 0 olan sətir:", (inst == 0).sum())

orders_ids = pd.read_sql("SELECT order_id FROM silver.orders", get_engine())["order_id"]
print("silver.orders-də olmayan order_id:", (~pay["order_id"].isin(orders_ids)).sum())


Ölçü: (103886, 5)
Boş mətn sayı:
 order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
Tam təkrar sətir: 0
(order_id, payment_sequential) təkrarı: 0

Ödəniş növləri:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3

Oxunmayan -> sequential: 0 | installments: 0 | value: 0
installments min/max: 0 24
value min/max: 0.0 13664.08
value = 0 olan sətir: 9
installments = 0 olan sətir: 2


silver.orders-də olmayan order_id: 0
